## CS260R Assignment 4: Imitation Learning & Preference Optimization

In this assignment, you will implement three imitation learning algorithms for autonomous driving in MetaDrive:

1. **Behavior Cloning (BC)**: Directly imitates demonstration actions via supervised learning
2. **HG-DAgger**: Fine-tunes BC with expert recovery data from dangerous situations
3. **Direct Preference Optimization (DPO)**: Learns from preference pairs to surpass simple imitation

The data consists of:
- **Demonstration data**: Rollouts from an underperforming driving agent
- **Recovery data**: Expert corrective actions recorded when the agent approached danger
- **Preference pairs**: Danger-aware pairs where a corrective (near-expert) action is preferred over a bad action that led toward road boundaries

### Setup

Please install the following dependencies:

```
pip install metadrive-simulator
pip install mediapy
pip install torch
pip install matplotlib
pip install tqdm
```

We tested with python==3.9 on Ubuntu 22.04.

In [ ]:
import subprocess, sys, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
from tqdm import tqdm
import mediapy as media

from metadrive.envs.metadrive_env import MetaDriveEnv
from metadrive.constants import TerminationState


# ===== Environment Setup =====

NUM_SCENARIOS = 100
env = MetaDriveEnv(dict(
    map=3, traffic_density=0.1, num_scenarios=NUM_SCENARIOS,
    start_seed=0, horizon=2000, use_render=False,
))
obs, _ = env.reset(seed=0)
state_dim = obs.shape[0]
action_dim = env.action_space.shape[0]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}, State dim: {state_dim}, Action dim: {action_dim}')

# If your find "AssertionError: Can not call this API after engine initialization!", please skip this code block because you already run it before.

In [ ]:
# ===== Dataset Classes =====

class DemoDataset:
    """Demonstration dataset for Behavior Cloning training.

    Attributes:
        states:  torch.Tensor [N, state_dim] - observation states
        actions: torch.Tensor [N, action_dim] - demonstration actions from a biased agent
    """
    def __init__(self, states: torch.Tensor, actions: torch.Tensor):
        self.states = states
        self.actions = actions

    def __len__(self):
        return len(self.states)

    @classmethod
    def load(cls, path='assignment4_data.pt'):
        data = torch.load(path)
        return cls(data['demo_states'], data['demo_actions'])


class RecoveryDataset:
    """Recovery dataset from expert interventions (HG-DAgger).

    Contains states and expert actions recorded when the expert took over
    to correct dangerous situations during data collection.

    Attributes:
        states:  torch.Tensor [N, state_dim] - states during expert recovery
        actions: torch.Tensor [N, action_dim] - expert corrective actions
    """
    def __init__(self, states: torch.Tensor, actions: torch.Tensor):
        self.states = states
        self.actions = actions

    def __len__(self):
        return len(self.states)

    @classmethod
    def load(cls, path='assignment4_data.pt'):
        data = torch.load(path)
        return cls(data['recov_states'], data['recov_actions'])


class PreferenceDataset:
    """Preference dataset for DPO training.

    Each entry is a preference pair (state, pos_action, neg_action) where
    pos_action is preferred over neg_action.

    Attributes:
        states:      torch.Tensor [N, state_dim] - states where danger was detected
        pos_actions: torch.Tensor [N, action_dim] - preferred actions (near-expert, corrective)
        neg_actions: torch.Tensor [N, action_dim] - dispreferred actions (biased, led to danger)
    """
    def __init__(self, states: torch.Tensor, pos_actions: torch.Tensor, neg_actions: torch.Tensor):
        self.states = states
        self.pos_actions = pos_actions
        self.neg_actions = neg_actions

    def __len__(self):
        return len(self.states)

    @classmethod
    def load(cls, path='assignment4_data.pt'):
        data = torch.load(path)
        return cls(data['pref_states'], data['pref_pos'], data['pref_neg'])
    
# ===== Policy Network =====

class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, action_dim), nn.Tanh(),
        )
    def forward(self, x):
        return self.net(x)

# ===== Evaluation Function =====

@torch.no_grad()
def evaluate(policy, n_episodes=10):
    """Evaluate policy and return (mean_reward, std_reward, success_rate)."""
    policy.eval()
    rewards, successes = [], []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=i % NUM_SCENARIOS)
        done, total_r = False, 0
        while not done:
            action = policy(torch.tensor(obs, dtype=torch.float32).to(device)).cpu().numpy()
            obs, r, tm, tr, info = env.step(action)
            total_r += r
            done = tm or tr
            if done:
                successes.append(float(info.get(TerminationState.SUCCESS, False)))
        rewards.append(total_r)
    policy.train()
    return np.mean(rewards), np.std(rewards), np.mean(successes)

# ===== Visualization Function =====

@torch.no_grad()
def visualize_episodes(policy, env, title='Agent', num_episodes=10, max_steps=2000):
    """Visualize policy rollouts. All episodes combined into one video."""
    policy.eval()
    all_rewards, all_successes, all_frames = [], [], []
    for ep in range(num_episodes):
        obs, _ = env.reset(seed=ep % NUM_SCENARIOS)
        done, total_r, step = False, 0, 0
        while not done and step < max_steps:
            frame = env.render(mode='topdown', window=False,
                               screen_record=False, screen_size=(400, 400))
            if frame is not None:
                all_frames.append(frame)
            action = policy(torch.tensor(obs, dtype=torch.float32).to(device)).cpu().numpy()
            obs, r, tm, tr, info = env.step(action)
            total_r += r
            done = tm or tr
            step += 1
            if done:
                success = info.get(TerminationState.SUCCESS, False)
                all_successes.append(float(success))
        all_rewards.append(total_r)
        status = 'SUCCESS' if all_successes[-1] else 'FAIL'
        print(f'{title} Ep {ep+1}/{num_episodes}: Return={total_r:.1f}, Steps={step}, {status}')
    avg_ret = np.mean(all_rewards)
    avg_sr = np.mean(all_successes)
    print(f'{title} Summary: Return={avg_ret:.1f}\u00b1{np.std(all_rewards):.1f}, SR={avg_sr:.2f}')
    if all_frames:
        media.show_video(all_frames, fps=30, title=title)
    policy.train()
    return all_rewards

## Step 1: Generate Preference Data

Run the data generation script to collect demonstration data and danger-aware preference pairs.

In [ ]:
# Generate data if not already cached
if not os.path.exists('assignment4_data.pt'):
    subprocess.run([sys.executable, 'generate_assignment4_data.py'], check=True)
    print('Data generation complete.')
else:
    print('Data already exists, skipping generation.')

## Step 2: Behavior Cloning

Behavior Cloning trains a policy to directly imitate the demonstration actions via supervised learning.

The BC loss is the mean squared error between predicted and target actions:

$$\mathcal{L}_{BC} = \mathbb{E}_{(s,a) \sim \mathcal{D}_{demo}} \left[ \| \pi_\theta(s) - a \|^2 \right]$$

This BC model trained on biased demonstration data serves as both a baseline and the initialization/reference for DPO training.

In [ ]:
# Load demonstration data for BC training
demo_data = DemoDataset.load('assignment4_data.pt')
print(f'Demonstration data: {len(demo_data)} samples')
print(f'  states shape:  {demo_data.states.shape}')
print(f'  actions shape: {demo_data.actions.shape}')

In [ ]:
########## CORE: Implement BC Loss ##########

def compute_bc_loss(policy, states, actions):
    """Compute Behavior Cloning loss (MSE between predicted and target actions).

    Args:
        policy: Policy network that maps states to actions
        states: Batch of states [B, state_dim]
        actions: Batch of target actions [B, action_dim]

    Returns:
        Scalar loss value
    """
    ##### Your code here #####
    pass
    ##########################

########## END CORE ##########

In [ ]:
# BC Training on demonstration data
BC_STEPS = 5000
BATCH_SIZE = 128
LR = 1e-4

torch.manual_seed(0)
bc_model = Policy().to(device)
optimizer = torch.optim.Adam(bc_model.parameters(), lr=LR)

bc_losses = []
for step in tqdm(range(BC_STEPS), desc='BC Training'):
    idx = np.random.choice(len(demo_data), BATCH_SIZE)
    loss = compute_bc_loss(bc_model, demo_data.states[idx].to(device), demo_data.actions[idx].to(device))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    bc_losses.append(loss.item())
    if (step + 1) % 1000 == 0:
        tqdm.write(f'  Step {step+1}: loss={loss.item():.4f}')

bc_checkpoint = copy.deepcopy(bc_model.state_dict())
print('BC training complete. Checkpoint saved.')

### Plot BC Loss Curve

Plot the training loss curve for Behavior Cloning. The loss values are stored in `bc_losses`.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

##### Your code here: plot bc_losses #####


##########################

In [ ]:
# Visualize BC agent
print('=' * 50)
print('BC Agent Evaluation')
print('=' * 50)
bc_model.load_state_dict(bc_checkpoint)
bc_rewards = visualize_episodes(bc_model, env, title='BC')
print('BC agent should have rewards around 20~100')

## Step 3: HG-DAgger

HG-DAgger (Human-Gated DAgger) improves upon Behavior Cloning by incorporating expert corrections at critical moments. During data collection, when the underperforming agent drives toward danger (road boundaries), an expert takes over to recover the car. These recovery transitions provide valuable corrective signals at states the agent actually visited.

HG-DAgger fine-tunes the BC model on this recovery data using the same BC loss:

$$\mathcal{L}_{HG} = \mathbb{E}_{(s,a) \sim \mathcal{D}_{recovery}} \left[ \| \pi_\theta(s) - a \|^2 \right]$$

Since the recovery data contains expert actions at dangerous states, this bridges the distribution gap between training and deployment.

In [ ]:
# Load recovery data (expert interventions during danger)
recov_data = RecoveryDataset.load('assignment4_data.pt')
print(f'Recovery data: {len(recov_data)} samples')
print(f'  states shape:  {recov_data.states.shape}')
print(f'  actions shape: {recov_data.actions.shape}')

# HG-DAgger Training: fine-tune BC model on recovery data
HG_STEPS = 5000

hg_model = Policy().to(device)
hg_model.load_state_dict(bc_checkpoint)  # Start from BC checkpoint
optimizer = torch.optim.Adam(hg_model.parameters(), lr=LR)

hg_losses = []
for step in tqdm(range(HG_STEPS), desc='HG-DAgger Training'):
    idx = np.random.choice(len(recov_data), BATCH_SIZE)
    ########## CORE: HG-DAgger Loss ##########
    ##### Your code here #####
    hg_loss = None  # Hint: call compute_bc_loss with hg_model and a batch from recov_data
    ##########################
    ########## END CORE ##########
    optimizer.zero_grad(); hg_loss.backward(); optimizer.step()
    hg_losses.append(hg_loss.item())
    if (step + 1) % 1000 == 0:
        tqdm.write(f'  Step {step+1}: loss={hg_loss.item():.4f}')

hg_checkpoint = copy.deepcopy(hg_model.state_dict())
print('HG-DAgger training complete. Checkpoint saved.')

In [ ]:
##### Your code here: plot hg_losses #####


##########################

In [ ]:
# Visualize HG-DAgger agent
print('=' * 50)
print('HG-DAgger Agent Evaluation')
print('=' * 50)
hg_model.load_state_dict(hg_checkpoint)
hg_rewards = visualize_episodes(hg_model, env, title='HG-DAgger')
print('HG-DAgger agent should improve upon BC (rewards around 200~320)')

## Step 4: Direct Preference Optimization (DPO)

DPO learns from preference pairs to improve beyond simple behavior cloning.

Given preference pairs $(s, a^+, a^-)$ where $a^+$ is preferred over $a^-$, the DPO loss is:

$$\mathcal{L}_{DPO} = -\mathbb{E} \left[ \log \sigma \left( \log \frac{\pi_\theta(a^+|s)}{\pi_{ref}(a^+|s)} - \log \frac{\pi_\theta(a^-|s)}{\pi_{ref}(a^-|s)} \right) \right]$$

where $\sigma(x) = \frac{1}{1+e^{-x}}$ is the sigmoid function.

For continuous actions, we approximate the log-probability using negative squared error:

$$\log \pi(a|s) \approx -\| \pi_\theta(s) - a \|^2$$

Substituting this approximation into the DPO loss gives:

$$\mathcal{L}_{DPO} = -\mathbb{E} \left[ \log \sigma \left( \underbrace{\left( \| \pi_\theta(s) - a^- \|^2 - \| \pi_\theta(s) - a^+ \|^2 \right)}_{\text{current policy prefers } a^+} - \underbrace{\left( \| \pi_{ref}(s) - a^- \|^2 - \| \pi_{ref}(s) - a^+ \|^2 \right)}_{\text{reference policy preference}} \right) \right]$$

**Training Setup:**
- **Reference policy $\pi_{ref}$**: Frozen BC model from Step 2 (trained on biased demo data)
- **DPO policy**: Initialized from the same BC checkpoint, then fine-tuned with DPO loss + BC regularization
- The policy learns from preference data to move toward positive (corrective) actions and away from negative (dangerous) actions

In [ ]:
# Load preference data for DPO training
pref_data = PreferenceDataset.load('assignment4_data.pt')
print(f'Preference pairs: {len(pref_data)} pairs')
print(f'  states shape:      {pref_data.states.shape}')
print(f'  pos_actions shape: {pref_data.pos_actions.shape}')
print(f'  neg_actions shape: {pref_data.neg_actions.shape}')

In [ ]:
########## CORE: Implement DPO Loss ##########

def compute_dpo_loss(policy, ref_policy, states, pos_actions, neg_actions):
    """Compute Direct Preference Optimization loss for continuous actions.

    Uses log π(a|s) ≈ -||π(s) - a||² as the log-probability approximation.

    Args:
        policy: The policy being trained
        ref_policy: Frozen reference policy (BC pretrained on demo data)
        states: States from preference pairs [B, state_dim]
        pos_actions: Preferred actions (near-expert) [B, action_dim]
        neg_actions: Dispreferred actions (biased, led to danger) [B, action_dim]

    Returns:
        Scalar DPO loss value
    """
    ##### Your code here #####
    pass
    ##########################

########## END CORE ##########

In [ ]:
# ===== DPO Training =====
# Fine-tune the BC model with DPO loss + BC regularization

DPO_STEPS = 5000
DPO_LR = 3e-5
BC_REG_WEIGHT = 0.5

# Frozen reference policy = BC model from Step 2
ref_model = Policy().to(device)
ref_model.load_state_dict(bc_checkpoint)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

# DPO policy initialized from the same BC checkpoint
dpo_model = Policy().to(device)
dpo_model.load_state_dict(bc_checkpoint)
optimizer = torch.optim.Adam(dpo_model.parameters(), lr=DPO_LR)

dpo_losses = []
bc_reg_losses = []
for step in tqdm(range(DPO_STEPS), desc='DPO Training'):
    # BC regularization on demonstration data
    idx = np.random.choice(len(demo_data), BATCH_SIZE)
    bc_loss = compute_bc_loss(dpo_model, demo_data.states[idx].to(device), demo_data.actions[idx].to(device))

    # DPO loss on preference data
    pidx = np.random.choice(len(pref_data), min(BATCH_SIZE, len(pref_data)))
    dpo_loss = compute_dpo_loss(dpo_model, ref_model,
                                pref_data.states[pidx].to(device),
                                pref_data.pos_actions[pidx].to(device),
                                pref_data.neg_actions[pidx].to(device))

    total_loss = BC_REG_WEIGHT * bc_loss + dpo_loss
    optimizer.zero_grad(); total_loss.backward(); optimizer.step()

    dpo_losses.append(dpo_loss.item())
    bc_reg_losses.append(bc_loss.item())

    if (step + 1) % 1000 == 0:
        tqdm.write(f'  Step {step+1}: dpo={dpo_loss.item():.4f}, bc_reg={bc_loss.item():.4f}')

dpo_checkpoint = copy.deepcopy(dpo_model.state_dict())
print('DPO training complete. Checkpoint saved.')

### Plot DPO Loss Curves

Plot the training loss curves for DPO. The DPO loss values are in `dpo_losses` and the BC regularization loss values are in `bc_reg_losses`.

In [ ]:
##### Your code here: plot dpo_losses and bc_reg_losses #####


##########################

In [ ]:
# Visualize DPO agent
print('=' * 50)
print('DPO Agent Evaluation')
print('=' * 50)
dpo_model.load_state_dict(dpo_checkpoint)
dpo_rewards = visualize_episodes(dpo_model, env, title='DPO')
print('DPO agent should have rewards around 250~370')

print(f'\n{"=" * 50}')
print(f'Final Comparison:')
print(f'  BC:       Return={np.mean(bc_rewards):.1f}\u00b1{np.std(bc_rewards):.1f}')
print(f'  HG-DAgger: Return={np.mean(hg_rewards):.1f}\u00b1{np.std(hg_rewards):.1f}')
print(f'  DPO:      Return={np.mean(dpo_rewards):.1f}\u00b1{np.std(dpo_rewards):.1f}')
print(f'  Gap: DPO - BC = {np.mean(dpo_rewards) - np.mean(bc_rewards):+.1f}')
print(f'{"=" * 50}')
